# Local UAT: Daily.dev Email Reader MCP Server

This notebook walks through end-to-end acceptance testing of the `news-agent` MCP server on your local machine.

**Before you start:**
- Python 3.10+ is installed.
- You have a Gmail account with 2FA and an App Password.
- You have at least one unread email from `informer@daily.dev` in your inbox (for the email UAT steps).

**Notebook kernel:** use the local `.venv` Python interpreter:
```bash
.venv\Scripts\python -m pip install ipykernel
.venv\Scripts\python -m ipykernel install --user --name news_agent --display-name "Python (news_agent)"
```
Then in Jupyter: `Kernel > Change kernel > Python (news_agent)`

## 1. Environment check

In [ ]:
import sys
print(sys.executable)
print(sys.version)

Install the package in editable mode with dev dependencies.

In [ ]:
!{sys.executable} -m pip install -e ".[dev]"

## 2. Configure environment

Create `.env` from the example. Replace the placeholder values with your real credentials.
For faster UAT failure when credentials are wrong, set `IMAP_TIMEOUT=10` (seconds).

In [ ]:
from pathlib import Path

env_path = Path(".env")
if not env_path.exists():
    env_path.write_text(
        "GMAIL_EMAIL=your.email@gmail.com\n"
        "GMAIL_APP_PASSWORD=xxxx xxxx xxxx xxxx\n"
        "MCP_API_TOKEN=your-secret-mcp-token\n"
        "HOST=0.0.0.0\n"
        "PORT=8000\n"
        "LOG_LEVEL=info\n"
        "IMAP_TIMEOUT=10\n"
    )
    print(".env created. Edit it with your credentials before continuing.")
else:
    print(".env already exists.")

print(env_path.read_text())

⚠️ **Stop here and edit `.env` with your real credentials if you have not already done so.**

Gmail-dependent steps will hang if the credentials are wrong or if IMAP (`imap.gmail.com:993`) is unreachable. The server now uses `IMAP_TIMEOUT` (default 30 s) and an endpoint-level `asyncio.wait_for` so it will fail fast instead of hanging forever.

## 3. Start the server

Launch the Uvicorn server in the background. The notebook will reuse this process for the remaining cells.

If you already have a server running from a previous test, stop it first (e.g. close the terminal or use Task Manager) so this cell starts a fresh process with the latest timeout settings.

In [ ]:
import subprocess
import time

# Kill any existing process listening on port 8000 (Windows)
!for /f "tokens=5" %a in ('netstat -ano ^| findstr :8000') do taskkill /F /PID %a 2>nul

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
time.sleep(3)  # give the server time to start
print(f"Server PID: {server.pid}")

## 4. Health check

The health endpoint is public and requires no token.

In [ ]:
import requests

BASE_URL = "http://localhost:8000"
REQUEST_TIMEOUT = 60  # seconds; increase if Gmail IMAP is slow

resp = requests.get(f"{BASE_URL}/health", timeout=REQUEST_TIMEOUT)
print(resp.status_code)
print(resp.json())
assert resp.status_code == 200
assert resp.json()["status"] == "ok"

## 5. Authentication check

A request without the Bearer token should fail with 401.

In [ ]:
resp = requests.post(f"{BASE_URL}/list-daily-dev-emails", timeout=REQUEST_TIMEOUT)
print(resp.status_code)
assert resp.status_code == 401

## 6. Load credentials

Load the token and basic-validate that the `.env` file was edited.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
MCP_TOKEN = os.getenv("MCP_API_TOKEN")
GMAIL_EMAIL = os.getenv("GMAIL_EMAIL", "")
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD", "")
headers = {"Authorization": f"Bearer {MCP_TOKEN}"}

if not MCP_TOKEN or "your-secret" in MCP_TOKEN:
    raise ValueError("MCP_API_TOKEN is not configured in .env")
if not GMAIL_EMAIL or "your.email" in GMAIL_EMAIL or not GMAIL_APP_PASSWORD:
    raise ValueError("Gmail credentials are not configured in .env")

print("Credentials loaded.")

## 7. List unread daily.dev emails

Requires a real Gmail connection and valid credentials.

If this cell hangs, it is almost always because the server cannot reach `imap.gmail.com:993` or the credentials in `.env` are wrong. The server will now return a `504` error after `IMAP_TIMEOUT + 5` seconds instead of hanging forever.

Expected result: JSON with `emails` array containing unread messages from `informer@daily.dev`.

In [ ]:
resp = requests.post(
    f"{BASE_URL}/list-daily-dev-emails",
    headers=headers,
    json={"limit": 10},
    timeout=REQUEST_TIMEOUT,
)
print(resp.status_code)
print(resp.json())
assert resp.status_code == 200

In [ ]:
resp.json()

## 8. Read articles from the latest unread daily.dev email

This fetches the most recent unread email, parses the articles, downloads each article's text, and marks the email as read.

⚠️ This will mark the email as read in your Gmail inbox.

In [ ]:
resp = requests.post(
    f"{BASE_URL}/read-daily-dev-articles",
    headers=headers,
    json={"uid": "54454"},  # use latest unread email
    timeout=REQUEST_TIMEOUT,
)
print(resp.status_code)
data = resp.json()
print(data.keys())
for article in data.get("articles", []):
    print("-" * 40)
    print(f"Author: {article['author']}")
    print(f"Header: {article['header']}")
    print(f"Link: {article['article_link']}")
    print(f"Text length: {len(article['article_text'] or '')}")
    print(f"Error: {article['error']}")

assert resp.status_code == 200
assert "email_uid" in data

## 9. Read a specific article by URL

This endpoint does not require Gmail and can be tested with any public article URL.

In [ ]:
article_url = "https://daily.dev/em/t/c?r=%2Fposts%2FsAEYsDtQ9%3Futm_source%3Dnotification%26utm_medium%3Demail%26utm_campaign%3Ddigest&link_id=eyJlbWFpbF9pZCI6IlJMbkNDUVVBQVo5anVZaEU5cTBvbjZEeUtIdElOdz09IiwiaHJlZiI6Imh0dHBzOi8vZGFpbHkuZGV2L2VtL3QvYz9yPSUyRnBvc3RzJTJGc0FFWXNEdFE5JTNGdXRtX3NvdXJjZSUzRG5vdGlmaWNhdGlvbiUyNnV0bV9tZWRpdW0lM0RlbWFpbCUyNnV0bV9jYW1wYWlnbiUzRGRpZ2VzdFx1MDAyNmxpbmtfaWQ9Q0lPLS1MSU5LSUQiLCJpbnRlcm5hbCI6ImI5YzIwOTcwOWQ1ZTllZThjZDAxIiwibGlua19pZCI6MTE0M30__76e8d5a98a76110b8de57c7f56b1247f2aba7846f96a60bfd9dc66672036cfbd"

resp = requests.post(
    f"{BASE_URL}/read-article-url",
    headers=headers,
    json={"url": article_url},
    timeout=REQUEST_TIMEOUT,
)
print(resp.status_code)
data = resp.json()
print(data.keys())
print(f"Text length: {len(data.get('article_text') or '')}")
print(f"Error: {data.get('error')}")

assert resp.status_code == 200
assert data["article_link"] == article_url

In [ ]:
data.get('article_text')

## 10. SSE transport smoke test

The MCP server exposes an SSE endpoint at `/sse`. This cell opens a short-lived SSE connection and prints the first few events.

In [ ]:
import httpx

with httpx.stream("GET", f"{BASE_URL}/sse", headers=headers, timeout=10) as response:
    print(f"SSE status: {response.status_code}")
    assert response.status_code == 200
    for line in response.iter_lines():
        print(line)
        # The MCP SSE endpoint sends an initial `endpoint` event and then waits
        # for the client to POST the initialize message to the messages endpoint.
        if line.startswith("data: "):
            assert "/sse/messages/" in line
            print("SSE endpoint is ready")
            break

## 11. Stop the server

Clean up the background Uvicorn process.

In [ ]:
server.terminate()
try:
    server.wait(timeout=5)
except subprocess.TimeoutExpired:
    server.kill()
print("Server stopped.")